In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/06 18:28:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'3.4.4'

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-06 18:34:10--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.226.36.130, 13.226.36.218, 13.226.36.196, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.226.36.130|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  60.9MB/s    in 1.0s    

2025-03-06 18:34:11 (60.9 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [5]:
!ls

001_spark_start.ipynb	       data
04_pyspark.ipynb	       download_data.sh
05_taxi_schema.ipynb	       fhvhv
06_spark_sql.ipynb	       fhvhv_tripdata_2021-01.csv
09_google_cloud_storage.ipynb  fhvhv_tripdata_2021-01.csv.gz
10_local_spark_cluster.ipynb   head.csv
10_local_spark_cluster.py      lib
11_spark_bq.py		       taxi_zone_lookup.csv
Untitled.ipynb		       week_5_homework.ipynb
Untitled1.ipynb		       yellow_tripdata_2024-10.parquet
Untitled2.ipynb		       zones


In [6]:
df = spark.read \
     .option("header", "true") \
     .parquet('yellow_tripdata_2024-10.parquet')

In [7]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-10-01 00:30:44|  2024-10-01 00:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

In [8]:
df = df.repartition(4)

In [9]:
df.write.parquet('yellow/2024/10')

In [10]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [11]:
df.registerTempTable('trips_data')

/home/kpivert/spark/spark-3.4.4-bin-hadoop3/python/pyspark/sql/dataframe.py:330: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [12]:
spark.sql("""
select * from trips_data limit 10;
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-10-07 16:40:43|  2024-10-07 18:10:56|              1|         14.8|        99|                 N|         127|         225|           1|       47.5|  0.0|    0.5|       0.

In [18]:
spark.sql("""
SELECT 
    COUNT(1)
FROM trips_data
WHERE DATE_TRUNC('day', tpep_pickup_datetime) = '2024-10-15 00:00:00';
""").show()

+--------+
|count(1)|
+--------+
|  128893|
+--------+



In [38]:
from pyspark.sql.functions import date_trunc, col, count, lit, unix_timestamp, desc

In [28]:
df.groupBy(date_trunc("day", col("tpep_pickup_datetime")).alias("date_truncated")) \
           .agg(count("*").alias("row_count")).show()

+-------------------+---------+
|     date_truncated|row_count|
+-------------------+---------+
|2024-10-13 00:00:00|   109969|
|2024-10-01 00:00:00|   119118|
|2024-09-30 00:00:00|       12|
|2024-10-14 00:00:00|   101513|
|2024-10-12 00:00:00|   130159|
|2024-11-01 00:00:00|       26|
|2024-10-10 00:00:00|   143206|
|2024-10-19 00:00:00|   135827|
|2024-10-17 00:00:00|   136330|
|2024-10-30 00:00:00|   132692|
|2024-10-04 00:00:00|   112431|
|2024-10-15 00:00:00|   128893|
|2024-10-11 00:00:00|   128821|
|2024-10-21 00:00:00|   107430|
|2024-10-24 00:00:00|   137337|
|2024-10-05 00:00:00|   124443|
|2024-10-08 00:00:00|   121402|
|2024-10-22 00:00:00|   121106|
|2024-10-09 00:00:00|   129915|
|2024-10-31 00:00:00|   129394|
+-------------------+---------+
only showing top 20 rows



In [39]:
df.withColumn(
    "hours_difference",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600
).select("hours_difference").orderBy(desc("hours_difference")).show()


+------------------+
|  hours_difference|
+------------------+
|162.61777777777777|
|           143.325|
|137.76055555555556|
|114.83472222222223|
| 89.89833333333333|
| 89.44611111111111|
| 70.29916666666666|
| 67.57333333333334|
| 66.06666666666666|
|           46.4225|
| 42.30888888888889|
| 38.47416666666667|
| 33.95111111111111|
| 26.29861111111111|
| 25.29138888888889|
|25.238333333333333|
|             24.47|
|23.996666666666666|
|23.995277777777776|
|23.994722222222222|
+------------------+
only showing top 20 rows



In [40]:
zones = spark.read \
     .option("header", "true") \
     .csv('taxi_zone_lookup.csv')

In [41]:
zones.createOrReplaceTempView("taxi_zone")

In [42]:
zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [47]:
spark.sql("""
select count(1),
PULocationID, 
Zone
from trips_data
left join taxi_zone on PULocationID = LocationID
group by PULocationID, Zone
order by count(1);
""").show()

+--------+------------+--------------------+
|count(1)|PULocationID|                Zone|
+--------+------------+--------------------+
|       1|         105|Governor's Island...|
|       2|           5|       Arden Heights|
|       2|         199|       Rikers Island|
|       3|           2|         Jamaica Bay|
|       3|         111| Green-Wood Cemetery|
|       4|         245|       West Brighton|
|       4|         204|   Rossville/Woodrow|
|       4|         187|       Port Richmond|
|       4|          44|Charleston/Totten...|
|       4|          84|Eltingville/Annad...|
|       6|          59|        Crotona Park|
|       6|         109|         Great Kills|
|       7|         118|Heartland Village...|
|       7|         156|     Mariners Harbor|
|       9|         176|             Oakwood|
|       9|         206|Saint George/New ...|
|      10|         172|New Dorp/Midland ...|
|      10|          30|       Broad Channel|
|      12|         184|     Pelham Bay Park|
|      12|